# W4W OSM publication threshold

## TL;DR

The defensible voluntary cutoff is **124 total ways**: pedestrian and cycling pathway types with at least 70% modeled spatial coverage, evidence from at least two distinct rides, a valid `intermediate` or `bad` proposal, and no pre-existing `smoothness` tag.

The first 100 are the top-ranked rows inside that tier. The cutoff admits **24 more ways**, then stops before rank 125 changes to motor/service-road types. This is an evidence tier, not a calibrated accuracy percentage.

## Scope and source grain

- Candidate grain: one OSM way.
- Evidence snapshot: classified traces from 173 rides, matched to the August 11, 2026 Tallinn OSM network snapshot.
- Publication state: batch manifests plus a fresh read-only OSM API check executed in this notebook.
- `bad_run_m` is the longest contiguous stretch of bad or very-bad evidence along a way. The dataset does not contain a general "uninterrupted ride length" metric.

In [1]:
from pathlib import Path
from collections import Counter
import csv, json, statistics, requests
import pandas as pd

ROOT = Path('../..').resolve()
MANIFEST_DIR = ROOT / 'pending_upload/osm-broad-2026-08'
SOURCE_CSV = ROOT / 'output/tallinn-coverage/osm-way-estimates.csv'
PATHWAY_TYPES = {'cycleway', 'footway', 'path', 'pedestrian'}

items = []
for path in sorted(MANIFEST_DIR.glob('batch-*.json')):
    items.extend(json.loads(path.read_text())['items'])
source = {int(row['way_id']): row for row in csv.DictReader(SOURCE_CSV.open())}
for item in items:
    row = source[item['way_id']]
    item['length_m'] = float(row['length_m'])
    item['covered_m'] = float(row['covered_m'])
    item['pathway'] = item['highway'] in PATHWAY_TYPES
    item['published'] = item.get('status') == 'pushed'

assert len(items) == 2398
assert len({item['way_id'] for item in items}) == 2398
assert sum(item['published'] for item in items) == 100
print('Loaded 2,398 unique live-preflight candidates and 100 published records.')

Loaded 2,398 unique live-preflight candidates and 100 published records.


## Fresh public OSM verification

This call is read-only. It checks whether each candidate currently has `smoothness=*` and verifies the published records against their logged changeset IDs.

In [2]:
def chunks(values, size=50):
    for index in range(0, len(values), size):
        yield values[index:index + size]

session = requests.Session()
session.headers['User-Agent'] = 'W4W-Tooling/0.1 publication-threshold-analysis'
live = {}
for batch in chunks([item['way_id'] for item in items]):
    response = session.get(
        'https://api.openstreetmap.org/api/0.6/ways.json',
        params={'ways': ','.join(map(str, batch))},
        timeout=30,
    )
    response.raise_for_status()
    for element in response.json().get('elements', []):
        live[int(element['id'])] = element

for item in items:
    element = live[item['way_id']]
    item['live_smoothness'] = (element.get('tags') or {}).get('smoothness')
    item['live_changeset'] = element['changeset']

exact = sum(
    item['published']
    and item['live_smoothness'] == item['proposed_value']
    and str(item['live_changeset']) == str(item['changeset_id'])
    for item in items
)
assert len(live) == 2398
assert exact == 100
assert Counter(item['live_smoothness'] for item in items) == Counter({None: 2298, 'intermediate': 76, 'bad': 24})
print('Fresh OSM result: 100/100 published ways match their logged tag and changeset; 2,298 candidates remain untagged.')

Fresh OSM result: 100/100 published ways match their logged tag and changeset; 2,298 candidates remain untagged.


## The natural cutoff is where pathway scope ends

The original sort puts the stronger evidence tier first, then pedestrian/cycling pathways before other highway types. Rank 124 is the last stronger-tier pathway. Rank 125 is a `service` way.

In [3]:
def cohort_row(label, rows):
    return {
        'cohort': label,
        'ways': len(rows),
        'published': sum(item['published'] for item in rows),
        'remaining_untagged': sum(item['live_smoothness'] is None for item in rows),
        'network_km': round(sum(item['length_m'] for item in rows) / 1000, 2),
        'median_coverage_pct': round(statistics.median(item['coverage_pct'] for item in rows), 1),
        'median_rides': statistics.median(item['rides'] for item in rows),
    }

cohorts = [
    cohort_row('Published top-ranked evidence', [item for item in items if item['published']]),
    cohort_row('Recommended pathway evidence tier', [item for item in items if item['confidence_tier']=='high_confidence' and item['pathway']]),
    cohort_row('All stronger-tier highway types', [item for item in items if item['confidence_tier']=='high_confidence']),
    cohort_row('Full broad pool', items),
]
pd.DataFrame(cohorts)

,cohort,ways,published,remaining_untagged,network_km,median_coverage_pct,median_rides
0,Published top-ranked evidence,100,100,0,8.09,100.0,3.0
1,Recommended pathway evidence tier,124,100,24,13.09,100.0,2.0
2,All stronger-tier highway types,311,100,211,29.54,97.7,2.0
3,Full broad pool,2398,100,2298,217.53,69.1,1.0


## What the 100 published ways look like

They are top-ranked within the stronger pathway tier, but they are not the output of a calibrated correctness model. The next 24 are all `bad` proposals because the sort places `intermediate` proposals first within the same pathway/evidence tier.

In [4]:
def distribution_row(label, rows):
    bad = [item for item in rows if item['proposed_value'] == 'bad']
    return {
        'cohort': label,
        'ways': len(rows),
        'intermediate': sum(item['proposed_value']=='intermediate' for item in rows),
        'bad': len(bad),
        'coverage_min': min(item['coverage_pct'] for item in rows),
        'coverage_median': round(statistics.median(item['coverage_pct'] for item in rows), 1),
        'rides_min': min(item['rides'] for item in rows),
        'rides_median': statistics.median(item['rides'] for item in rows),
        'way_length_median': round(statistics.median(item['length_m'] for item in rows), 1),
        'bad_run_median': round(statistics.median(item['bad_run_m'] for item in bad), 1) if bad else None,
        'mean_match_distance_median': round(statistics.median(item['mean_match_distance_m'] for item in rows), 2),
    }

published = [item for item in items if item['published']]
next_24 = [item for item in items if 101 <= item['rank'] <= 124]
pd.DataFrame([distribution_row('Published 100', published), distribution_row('Next 24', next_24)])

,cohort,ways,intermediate,bad,coverage_min,coverage_median,rides_min,rides_median,way_length_median,bad_run_median,mean_match_distance_median
0,Published 100,100,76,24,70.6,100.0,2,3.0,55.5,46.0,1.15
1,Next 24,24,0,24,70.5,89.2,2,2.0,171.4,46.3,0.90


## Exact remaining set under the recommendation

These 24 should still receive the same map review and create-only live check before any write.

In [5]:
pd.DataFrame([{
    'rank': item['rank'],
    'way_id': item['way_id'],
    'highway': item['highway'],
    'proposal': item['proposed_value'],
    'rides': item['rides'],
    'coverage_pct': item['coverage_pct'],
    'way_length_m': item['length_m'],
    'bad_run_m': item['bad_run_m'],
    'mean_match_distance_m': item['mean_match_distance_m'],
} for item in next_24])

,rank,way_id,highway,proposal,rides,coverage_pct,way_length_m,bad_run_m,mean_match_distance_m
0,101,979875286,cycleway,bad,3,100.0,160.0,30.9,2.17
1,102,705057503,footway,bad,3,100.0,36.7,34.2,0.66
2,103,1523614704,footway,bad,3,97.6,165.6,57.7,0.38
3,104,213878646,cycleway,bad,3,86.8,425.9,87.1,0.75
4,105,1190035113,path,bad,3,81.8,92.5,49.4,0.06
5,106,1124095629,pedestrian,bad,3,81.8,124.8,32.6,1.35
6,107,987191897,path,bad,2,100.0,38.6,38.6,0.57
7,108,987906369,cycleway,bad,2,100.0,101.7,33.5,0.55
8,109,592149154,cycleway,bad,2,99.5,157.9,37.4,1.61
9,110,229814879,footway,bad,2,99.0,101.7,89.9,0.73


## Decision

Use a transparent **conservative pathway evidence tier**, not “confidence score”: at least 70% modeled way coverage, at least two distinct rides, pathway type `cycleway`, `footway`, `path`, or `pedestrian`, a valid proposal under the existing rule, no prior `smoothness`, plus human review and a fresh create-only check.

This gives 124 total published candidates, 24 still eligible. Stop there unless a future validation study establishes a measured accuracy threshold. The threshold narrows quality and scope; it cannot guarantee that OSM reviewers will not review the edits.